# 00 · Setup check

Run all cells of this notebook before the workshop. It checks that the Python packages are installed and that a tiny spiking network runs on the Brian2 simulator. It takes about 10 seconds and needs no SpiNNaker2 hardware.

If a cell fails, see the install guide in the [README](README.md) or bring the error message to the workshop.

## 1. Packages

In [ ]:
import importlib

expected = {
    "spinnaker2": None,  # py-spinnaker2, installed from GitLab at v0.8.2
    "brian2": None,
    "numpy": "2.3.4",
    "torch": None,
    "snntorch": "0.9.4",
    "nir": "1.0.4",
    "nirtorch": "2.0.2",
    "matplotlib": None,
}
problems = []
for name, version in expected.items():
    try:
        module = importlib.import_module(name)
        found = getattr(module, "__version__", "?")
        status = "ok" if version is None or found == version else f"expected {version}"
        if status != "ok":
            problems.append(name)
        print(f"{name:12s} {found:12s} {status}")
    except ImportError as error:
        problems.append(name)
        print(f"{name:12s} MISSING      {error}")
print("\nAll packages ok." if not problems else f"\nCheck these packages: {problems}")

## 2. A tiny network on Brian2

One input neuron sends three spikes to one LIF neuron. The LIF neuron should answer each input spike with one spike of its own, a few steps later.

In [ ]:
from spinnaker2 import brian2_sim, snn

stimulus = snn.Population(size=1, neuron_model="spike_list", params={0: [1, 5, 9]}, name="input")
neuron = snn.Population(
    size=1,
    neuron_model="lif",
    params={"threshold": 1.0, "alpha_decay": 0.9, "reset": "reset_to_v_reset", "v_reset": 0.0},
    name="lif",
    record=["spikes"],
)
connection = snn.Projection(pre=stimulus, post=neuron, connections=[[0, 0, 2, 1]])  # pre, post, weight, delay

network = snn.Network("setup_check")
network.add(stimulus, neuron, connection)
brian2_sim.Brian2Backend().run(network, 20)

spikes = neuron.get_spikes()[0]
print("LIF spike times:", spikes)
print("Brian2 works." if len(spikes) > 0 else "No spikes: something is wrong, please bring this notebook to the workshop.")